# Survey

The survey is where everything comes together. It takes a scene, a scanner and a platform and the so-called "legs" that define the positions/trajectories of the platform during the survey and the speicifc settings of the scanner.

Let's demonstrate this for a basic ALS survey and prepare the platform, scanner and scene.

In [1]:
import helios
platform = helios.platform_from_name("sr22")
scanner = helios.scanner_from_name("leica_als50")
sceneparts = helios.ScenePart.from_objs("../data/sceneparts/toyblocks/*.obj")
scene = helios.StaticScene(sceneparts)

From this, we can create an "empty" survey, which we can then populate with legs and settings:

In [2]:
survey = helios.Survey(scene=scene, scanner=scanner, platform=platform)

In [3]:
# no legs defined yet
survey.legs

()

To use surveys created for ealier versions of `helios` (< v3.0.0), we can also load surveys from XML files. To make sure the survey can be loaded correctly, add the directory where the survey XML file is located as an asset directory (here: the root directory of the cloned `helios` repository):

In [4]:
helios.add_asset_directory("D:/Software/_helios_versions/helios/")
tls_toyblocks_survey = helios.Survey.from_xml("data/surveys/toyblocks/tls_toyblocks.xml")
tls_toyblocks_survey.scanner

## Scanner settings

Scanner settings are defined per leg. Often, many or all legs of a survey will share the same scanner settings. We can define them once and then pass them to the legs. Depending on the `optics` of the respective scanner, different scan settings are relevant.

Note that we always specify the unit of the settings, resolving any ambiguity and ensuring that the settings are correctly interpreted. This is done by the help of the package [pint](https://pint.readthedocs.io/en/stable/). We can either specify the units by multiplying with the unit from `helios.units` or by using a string with the unit.

In [5]:
scan_settings = helios.ScannerSettings(
    pulse_frequency=100_000 * helios.units.Hz,
    scan_angle = 20 * helios.units.deg,
    scan_frequency="70 Hz",
    trajectory_time_interval=0.01 * helios.units.s
)

## Platform settings

Likewise, we also have platform settings. We can use `PlatformSettings` for static platforms and `DynamicPlatformSettings` for moving platforms. The latter allows us to specify the speed of the platform, or - if using interpolated trajectory, the trajectory settings. However, since platform settings are usually not shared between legs, we will often just specify the individual parameters directly when constructing or adding legs.

## Legs

Legs are the positions or waypoints of a survey. In trajectory mode, they can also be portions of a given trajectory.

We define a list of waypoints that the airplane should fly through during the survey. Additionally, we set the speed of the platform.

In [6]:
waypoints = [
    [-50, -25, 250],
    [50, -25, 250],
    [50, 25, 250],
    [-50, 25, 250]
]
speed = 30  # m/s

Using a simple loop, we now add the legs at the respective positions and apply the scan settings.

In [7]:
for x, y, z in waypoints:
    survey.add_leg(x=x, y=y, z=z, speed_m_s=speed, scanner_settings=scan_settings)

## Running a survey

When running a survey, we can provide `ExecutionSettings` and `OutputSettings` to control the execution of the survey and the output format.

### Execution settings

For example, we can explicitly change the number of threads, the logging verbosity or the parallelization strategy.

In [8]:
execution_settings = helios.ExecutionSettings(
    num_threads=4,
    verbosity=helios.LogVerbosity.VERY_VERBOSE,
    parallelization=helios.ParallelizationStrategy.WAREHOUSE
)

In [9]:
pc, traj = survey.run(execution_settings=execution_settings)

### Output settings

The output settings allow to change the format in which the output is returned (incl. requesting waveform and pulse output), the directory where the output is written (if written to file), and more. We show a few options below.

In [10]:
# default
output_settings = helios.OutputSettings(
    format=helios.OutputFormat.NPY
)
# return a laspy object
output_settings = helios.OutputSettings(
    format=helios.OutputFormat.LASPY
)
# write to LAZ and change the las scale
output_settings = helios.OutputSettings(
    format=helios.OutputFormat.LAZ,
    las_scale=0.00025
)
# write to LAS and also write waveform and pulse output, use a custom output directory
output_settings = helios.OutputSettings(
    format=helios.OutputFormat.LAS,
    output_dir="detailed_output",
    write_waveform=True,
    write_pulse=True
)
# write to XYZ and split by channel (relevant for multi-channel devices)
output_settings = helios.OutputSettings(
    format=helios.OutputFormat.XYZ,
    split_by_channel=True
)

## Output formats

Note that depending on the chosen output format, the `survey.run()` method has either one of two returns:
- `OutputFormat.NPY` / `OutputFormat.LASPY`: two returns, a points array (or LASPY object) and a trajectory array
- `OutputFormat.LAS` / `OutputFormat.LAZ` / `OutputFormat.XYZ`: one return, the path to the output folder

Below is a demonstration of the different output formats. Here you can also see that the individual parameters of `ExecutionSettings` and `OutputSettings` can be passed directly to the `survey.run()` method, without explicitly creating the settings objects.


In [11]:
# output to arrays: two returns
pc, traj = survey.run(format=helios.OutputFormat.NPY)
las, traj2 = survey.run(format=helios.OutputFormat.LASPY, parallelization=helios.ParallelizationStrategy.WAREHOUSE)

# output to file: one return
outpath_laz = survey.run(format=helios.OutputFormat.LAZ, verbosity=helios.LogVerbosity.VERY_VERBOSE)


This is what the output looks like in each case:

In [12]:
pc[:10]

array([(0, 1, [-24.2681    ,  17.56639626,   9.84972579], [-4.49147234e-13,  1.74036743e-01, -9.84739159e-01], [-61.073084 , -27.807459 , 230.4609125], 244.58281354, 6.62253898, 0., 1, 1, 919110, 0, 200905.85774, 0),
       (0, 1, [-24.2678    ,  17.51864444,   8.63402369], [-4.46472962e-13,  1.73000509e-01, -9.84921735e-01], [-61.072784 , -27.807459 , 230.4609125], 245.77178848, 2.51876412, 0., 1, 1, 919111, 0, 200905.85775, 0),
       (0, 1, [-24.2675    ,  17.49672955,   7.25576214], [-4.43798194e-13,  1.71964083e-01, -9.85103220e-01], [-61.072484 , -27.807459 , 230.4609125], 247.12561374, 1.38474619, 0., 1, 1, 919112, 0, 200905.85776, 0),
       (0, 1, [-24.2672    ,  17.49289917,   5.75658591], [-4.41122936e-13,  1.70927467e-01, -9.85283615e-01], [-61.072184 , -27.807459 , 230.4609125], 248.601936  , 1.37794712, 0., 1, 1, 919113, 0, 200905.85777, 0),
       (0, 1, [-24.2669    ,  17.49489133,   4.20534711], [-4.38447189e-13,  1.69890661e-01, -9.85462918e-01], [-61.071884 , -27.807

In [13]:
pc["position"][:10]

array([[-24.2681    ,  17.56639626,   9.84972579],
       [-24.2678    ,  17.51864444,   8.63402369],
       [-24.2675    ,  17.49672955,   7.25576214],
       [-24.2672    ,  17.49289917,   5.75658591],
       [-24.2669    ,  17.49489133,   4.20534711],
       [-24.2666    ,  17.50617371,   2.58082298],
       [-24.2663    ,  17.49608343,   1.06174746],
       [-23.9555    ,  17.20432845,   5.89077783],
       [-23.9552    ,  17.19835672,   7.45442799],
       [-23.9549    ,  17.20421032,   8.93147037]])

In [14]:
print(las)
print(las.x)
print(las.gps_time)

<LasData(1.4, point fmt: <PointFormat(6, 12 bytes of extra dims)>, 43847 points, 1 vlrs)>
<ScaledArrayView([-24.2  -24.2  -24.2  ... -18.55 -18.55 -18.55])>
[200905.86001 200905.86002 200905.86003 ... 200912.28493 200912.28494
 200912.28495]


In [15]:
# the trajectory output is the same in both NPY and LASPY cases
traj[:10]

array([(200905.  , [-50.0003, -25.    , 250.    ], -0., 0., 4.71238898),
       (200905.01, [-49.7003, -25.    , 250.    ], -0., 0., 4.71238898),
       (200905.02, [-49.4003, -25.    , 250.    ], -0., 0., 4.71238898),
       (200905.03, [-49.1003, -25.    , 250.    ], -0., 0., 4.71238898),
       (200905.04, [-48.8003, -25.    , 250.    ], -0., 0., 4.71238898),
       (200905.05, [-48.5003, -25.    , 250.    ], -0., 0., 4.71238898),
       (200905.06, [-48.2003, -25.    , 250.    ], -0., 0., 4.71238898),
       (200905.07, [-47.9003, -25.    , 250.    ], -0., 0., 4.71238898),
       (200905.08, [-47.6003, -25.    , 250.    ], -0., 0., 4.71238898),
       (200905.09, [-47.3003, -25.    , 250.    ], -0., 0., 4.71238898)],
      dtype=[('gps_time', '<f8'), ('position', '<f8', (3,)), ('roll', '<f8'), ('pitch', '<f8'), ('yaw', '<f8')])

In [16]:
traj2[:10]

array([(200905.  , [-50.0003, -25.    , 250.    ], 0., 0., 4.71238898),
       (200905.01, [-49.7003, -25.    , 250.    ], 0., 0., 4.71238898),
       (200905.02, [-49.4003, -25.    , 250.    ], 0., 0., 4.71238898),
       (200905.03, [-49.1003, -25.    , 250.    ], 0., 0., 4.71238898),
       (200905.04, [-48.8003, -25.    , 250.    ], 0., 0., 4.71238898),
       (200905.05, [-48.5003, -25.    , 250.    ], 0., 0., 4.71238898),
       (200905.06, [-48.2003, -25.    , 250.    ], 0., 0., 4.71238898),
       (200905.07, [-47.9003, -25.    , 250.    ], 0., 0., 4.71238898),
       (200905.08, [-47.6003, -25.    , 250.    ], 0., 0., 4.71238898),
       (200905.09, [-47.3003, -25.    , 250.    ], 0., 0., 4.71238898)],
      dtype=[('gps_time', '<f8'), ('position', '<f8', (3,)), ('roll', '<f8'), ('pitch', '<f8'), ('yaw', '<f8')])

In [17]:
outpath_laz

WindowsPath('d:/Software/_helios_versions/helios/doc/output/2026-03-17_08-48-38')